Nilufer Belediyesi Acik Veri Portali (CKAN) - Arsa Birim Degerleri

Bu notebook, `acikveri.nilufer.bel.tr` uzerindeki CKAN API'sini kullanarak
'Arsa Birim Degerleri' veri setinin en guncel kaynak dosyasini bulur, gecici
olarak indirir, 2010 ve sonrasi ile filtreler, Spark DataFrame'e cevirir ve
data lake'e (bronze katman) yazar. Gecici dosya, veri okunduktan sonra hemen
silinir; localde kalici veri tutulmaz.

Maliyet notu: Kaynak dosya (XLSX) 1986-2026 arasi ~180K satir icerir ve tek
parca oldugu icin indirme/parse maliyeti azalmaz, ancak 2010 oncesi satirlar
(~%46) filtrelenerek data lake'e yazilan/depolanan veri hacmi ve sonraki
Spark islemlerinin maliyeti dusurulur.

Nazik/performansli yaklasim:
- Sadece 1 API cagrisi (`package_show`) yapilir, sadece metadata icin.
- Asil veri, `/api/` altinda olmayan resmi kaynak indirme URL'inden tek
  seferde indirilir (robots.txt'teki `Disallow: /api/` kapsamina girmez).
- Paralel/tekrarli istek yoktur, `Crawl-Delay: 10` kuralina saygi gosterilir.
- Lisans: CC BY 4.0 (Nilufer Belediyesi Acik Veri Lisansi) - kullanirken atif gerekir.

CKAN/dosya indirme fonksiyonlari (`get_ckan_resource`, `download_file`)
`Utils` notebook'undan gelir.

In [ ]:
%pip install pandas requests openpyxl

In [ ]:
%run "./Utils"

In [ ]:
import os
import time

import pandas as pd

CKAN_BASE_URL = "https://acikveri.nilufer.bel.tr"
DATASET_ID = "2026-arsa-birim-degerleri"

REQUEST_HEADERS = {
    "User-Agent": "aXet-Project/1.0 (acik veri arastirma amacli)",
    "Accept": "application/json",
}

In [ ]:
resource = get_ckan_resource(CKAN_BASE_URL, DATASET_ID, REQUEST_HEADERS)

print("Bulunan kaynak:", resource["name"])
print("Format:", resource["format"])
print("Son guncelleme:", resource.get("last_modified"))

temp_path = "/tmp/arsa_birim_degerleri.xlsx"

download_file(resource["url"], temp_path, request_headers=REQUEST_HEADERS)

time.sleep(1)

print("Gecici olarak indirildi:", temp_path)

In [ ]:
df_raw = pd.read_excel(temp_path)

os.remove(temp_path)

print("Gecici dosya silindi:", temp_path)

df = df_raw[["Mahalle Adı", "Cadde Sokak Adı", "Nitelik Adı", "Yıl", "Arsa Birim Değeri"]].copy()

df.columns = ["mahalle_adi", "cadde_sokak_adi", "nitelik_adi", "yil", "arsa_birim_degeri"]

df["yil"] = pd.to_numeric(df["yil"], errors="coerce")
df["arsa_birim_degeri"] = pd.to_numeric(df["arsa_birim_degeri"], errors="coerce")

MIN_YIL = 2010

row_count_before = len(df)

df = df[df["yil"] >= MIN_YIL].reset_index(drop=True)

print(f"Filtre oncesi satir: {row_count_before}")
print(f"Filtre sonrasi satir ({MIN_YIL}+): {len(df)}")

print(df.shape)
df.head(10)

In [ ]:
df_spark = spark.createDataFrame(df)

write_to_datalake(
    df_spark,
    "abfss://axetproject@ozandatalake001.dfs.core.windows.net/axet_bronze/nilufer_arsa_birim_degerleri/"
)